# Social Media Post Generation & Upload Workflow with Human-in-the-Loop (HITL)

This notebook implements an automated social media content workflow using LangGraph:
1. **Title & Description Generation**: Analyzes the topic and generates a catchy title and structured description using web search when needed.
2. **Streaming Post Generation**: Drafts a narrative-driven professional post and streams tokens in real time.
3. **Automated Quality Evaluation**: Evaluates the draft against clarity, tone, hook/CTA, and factual integrity criteria.
4. **Human-in-the-Loop (HITL) Review**: Pauses execution via LangGraph's `interrupt()` to show the complete draft to a human reviewer.
   - If **approved**: Passes to the upload node and prints/publishes the final post.
   - If **rejected**: Collects human feedback and loops back to regenerate the post addressing all critique points.


## 1. Imports & Environment Configuration


In [ ]:
import os
import sys
from pathlib import Path
from typing import Annotated, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from rich import print

# LangChain & LangGraph imports
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.types import Command, interrupt

# Robust discovery and loading of .env file from current or parent directories
env_path = None
for parent in [Path.cwd()] + list(Path.cwd().parents):
    candidate = parent / ".env"
    if candidate.is_file():
        env_path = candidate
        break

if env_path:
    load_dotenv(env_path)

openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")


## 2. State & Data Schemas


In [ ]:
class PostReview(BaseModel):
    """Stores a single rejected post draft along with reviewer feedback."""
    post_generated: str | None = Field(
        default=None,
        description="The content of the generated post draft."
    )
    feedback: str | None = Field(
        default=None,
        description="Actionable critique explaining why the post was rejected."
    )


class State(BaseModel):
    """The central state of the social media post generation and review workflow."""
    
    # Complete conversation message history
    messages: Annotated[list[BaseMessage], add_messages] = Field(
        default_factory=list,
        description="Chronological log of messages exchanged during generation."
    )
    
    # Original user request / topic
    user_query: str | None = Field(
        default=None,
        description="The original topic or prompt provided by the user."
    )
    
    # History of rejected drafts and corresponding critique
    rejected_posts: list[PostReview] = Field(
        default_factory=list,
        description="Archive of rejected post drafts paired with their feedback."
    )
    
    # Current active draft of the post
    post: str | None = Field(
        default=None,
        description="The most recently generated post content."
    )
    
    # Active feedback to incorporate into the next generation attempt
    feedback: str | None = Field(
        default=None,
        description="Critique from the evaluator or human reviewer for revision."
    )
    
    # Status flags
    qualified: bool = Field(
        default=False,
        description="True if the post passed automated editorial evaluation."
    )
    human_approved: bool = Field(
        default=False,
        description="True if the human reviewer explicitly approved the post."
    )
    
    # Counter to avoid infinite revision loops
    retry_count: int = Field(
        default=0,
        description="Number of regeneration iterations completed."
    )


## 3. Models and Tools Setup


In [ ]:
# Base chat model
model = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=openai_api_key,
    temperature=0
)

# Schema for structured title and description generation
class PostOutputSchema(BaseModel):
    """Structured output containing title and high-level description for the post."""
    title: str | None = Field(
        default=None,
        description="Catchy and engaging title for the post."
    )
    description: str | None = Field(
        default=None,
        description="Concise description outlining the theme and core message."
    )

# Search tool for fetching real-time facts and updates
search_tool = TavilySearch(max_results=3)
tools = [search_tool]
tool_by_name = {tool.name: tool for tool in tools}

# Model bound with search tool for content drafting
model_with_tools = model.bind_tools(tools)

# Model bound with search tool AND structured schema for initial planning
model_with_tools_and_schema = model.bind_tools([search_tool, PostOutputSchema])


## 4. Workflow Nodes: Generation & Evaluation


In [ ]:
def post_generation_node(state: State):
    """
    Step 1 & 2: Content Generation Agent.
    - Step 1: Generates a catchy title and structured description using web search if needed.
    - Step 2: Streams the post content in real time, weaving facts into a narrative.
    - If previous feedback exists (from evaluator or human), explicitly incorporates it.
    """
    user_query = state.user_query
    if not user_query:
        print("[bold red]No user_query found in state.[/bold red]")
        return {}
    
    # Display iteration status
    if state.feedback:
        print(f"\n[bold magenta]--- Regenerating Post (Attempt {state.retry_count + 1}) ---[/bold magenta]")
        print(f"[bold yellow]Addressing Feedback:[/bold yellow] {state.feedback}\n")
    else:
        print("[bold green]--- Step 1: Generating Title and Description ---[/bold green]")
    
    # --- Step 1: Generate Title and Description ---
    system_prompt_for_title_and_description = """You are a title and description generation agent.
        For the given user query your job is to analyse it first. Understand the tone and theme regarding
        which the title and description is supposed to be generated, and then generate the title which should
        be catchy in nature as well as generate a description which should be simple in tone, easy to understand,
        factual to the point and should contain an opening and closing note.
        If you need recent facts or updates to write an accurate description, use the tavily_search tool.
        When you have the required information, call the PostOutputSchema tool with the title and description.
    """
    title_messages = [
        SystemMessage(content=system_prompt_for_title_and_description),
        HumanMessage(content=user_query)
    ]

    title = None
    description = None

    while True:
        response = model_with_tools_and_schema.invoke(title_messages)
        title_messages.append(response)

        # Extract structured output if called
        schema_called = False
        for tool_call in response.tool_calls:
            if tool_call["name"] == "PostOutputSchema":
                title = tool_call["args"].get("title")
                description = tool_call["args"].get("description")
                schema_called = True
                break

        if schema_called:
            break
        
        # Execute external web search if requested by the model
        if response.tool_calls:
            for tool_call in response.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]
                tool_call_id = tool_call["id"]
                if tool_name in tool_by_name:
                    print(f"[bold cyan]Searching web for context:[/bold cyan] {tool_args.get('query')}")
                    tool_result = tool_by_name[tool_name].invoke(tool_args)
                    title_messages.append(
                        ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
                    )
        else:
            break

    print(f"[bold yellow]Title:[/bold yellow] {title}")
    print(f"[bold yellow]Description:[/bold yellow] {description}\n")
    
    # --- Step 2: Stream Post Content ---
    print("[bold green]--- Step 2: Generating Content (Streaming) ---[/bold green]\n")

    # Inject previous feedback if revising an earlier draft
    feedback_context = ""
    if state.feedback:
        feedback_context = f"""
        PREVIOUS FEEDBACK:
        The previous post draft received the following feedback:
        "{state.feedback}"

        MANDATORY REVISION REQUIREMENT:
        You must revise the post to specifically address and resolve this feedback while maintaining factual accuracy, professionalism, and engaging narrative flow.
        """

    system_prompt_for_content = f"""Act as an expert content writer and corporate storyteller. Your task is to write a formal,
        factual, and easy-to-understand post based on the provided user_query: {user_query} having title: {title} and description: {description}.
        The goal is to make the post highly impactful by weaving the facts into a compelling narrative arc. 
        You have access to tools to search the current updates regarding the topic if needed.
        {feedback_context}
        Structure & Key Requirements:
        Impactful Opening: Start with a powerful hook, a compelling question, or a striking fact that immediately captures the reader's attention and sets the tone.
        Storytelling Narrative: Present the core information smoothly. Take the reader from a problem or context to the solution or realization, keeping the language formal, clear, and strictly grounded in facts. Avoid jargon or dense fluff.
        Impactful Closing Line: Conclude with a memorable final thought, a strong call to action, or a resonant statement that leaves a lasting impression on the reader.

        Tone & Style:
        Tone: Formal, professional, and authoritative yet accessible.
        Clarity: Simple to understand (avoid overly complex sentences).
        Accuracy: 100% factual and to the point.
    """

    if state.feedback:
        human_content = f"Regenerate the post for query '{user_query}' with title '{title}' and description '{description}'. Ensure you incorporate this feedback: {state.feedback}"
    else:
        human_content = f"Generate a post having the user query '{user_query}', with title '{title}' and description '{description}'."

    content_messages = [
        SystemMessage(content=system_prompt_for_content),
        HumanMessage(content=human_content)
    ]
    
    final_post_content = ""
    while True:
        accumulated_response = None

        # Stream smoothly to stdout to preserve clean continuous text in Jupyter
        for chunk in model_with_tools.stream(content_messages):
            if accumulated_response is None:
                accumulated_response = chunk
            else:
                accumulated_response += chunk
            if chunk.content:
                sys.stdout.write(chunk.content)
                sys.stdout.flush()

        content_messages.append(accumulated_response)

        # Generation complete if no tool calls were made
        if not accumulated_response.tool_calls:
            final_post_content = accumulated_response.content
            sys.stdout.write("\n\n")
            sys.stdout.flush()
            break

        # Handle tool calls during drafting if necessary
        for tool_call in accumulated_response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]
            if tool_name in tool_by_name:
                print(f"\n[bold cyan]Searching web for post content:[/bold cyan] {tool_args.get('query')}")
                tool_result = tool_by_name[tool_name].invoke(tool_args)
                content_messages.append(
                    ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
                )

    # Return updated state (reset pass/approval flags for the new draft)
    return {
        "post": final_post_content,
        "messages": [AIMessage(content=final_post_content)],
        "qualified": False,
        "human_approved": False
    }


In [ ]:
class PostEvaluationSchema(BaseModel):
    """Schema for automated evaluation verdict and feedback."""
    qualified: bool = Field(
        description="True if the post meets all editorial quality criteria, False if revisions are required."
    )
    feedback: str = Field(
        default="",
        description="Constructive and actionable critique explaining what to improve if qualified is False."
    )

# Evaluator LLM bound with structured output
post_evaluator_model = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=openai_api_key,
    temperature=0
).with_structured_output(PostEvaluationSchema)


def post_evaluator_node(state: State):
    """
    Step 3: Post Quality Evaluator Agent.
    Evaluates the draft against 4 objective criteria:
    1. Clarity & Readability
    2. Tone & Professionalism
    3. Impactful Hook & Closing
    4. Factual Integrity
    """
    post = state.post or ""
    if not post:
        print("[bold red]No post found to evaluate.[/bold red]")
        return {"qualified": False, "feedback": "No post content was provided for evaluation."}

    print("[bold blue]--- Step 3: Evaluating Post Quality ---[/bold blue]")

    system_prompt = """You are an expert Post Evaluator Agent specializing in professional content review and editorial optimization.
Your task is to evaluate a given social media or professional post against specific qualitative criteria and provide a structured, constructive assessment.

Evaluation Criteria:
1. Clarity & Readability: Is the post easy to understand? Does it flow logically without jargon overload or convoluted sentence structures?
2. Tone & Professionalism: Is the tone appropriately formal, polished, and suited for a professional audience (e.g., LinkedIn, corporate communications)?
3. Impactful Hook & Closing: Does the opening line immediately grab attention, spark curiosity, or establish clear value? Does the closing line leave a lasting impression, drive engagement, or provide a strong concluding thought?
4. Factual Integrity: Are the claims, data points, and assertions logically sound, verifiable, and free of misleading hyperbole?

If the post satisfies all criteria to a high standard, set qualified to True and provide positive confirmation.
If the post has notable shortcomings, set qualified to False and provide detailed, actionable feedback on what specifically needs to be improved."""

    system_message = SystemMessage(content=system_prompt)
    human_message = HumanMessage(content=f"Please evaluate the following post:\n\n{post}")

    response: PostEvaluationSchema = post_evaluator_model.invoke([
        system_message,
        human_message
    ])

    if response.qualified:
        print("[bold green]✓ Post Passed Evaluation Criteria![/bold green]\n")
        return {
            "qualified": True,
            "feedback": None
        }
    else:
        print("[bold red]✗ Post Failed Evaluation Criteria.[/bold red]")
        print(f"[bold yellow]Evaluator Feedback:[/bold yellow] {response.feedback}\n")
        
        # Log rejected draft and feedback to history
        rejected_entry = PostReview(post_generated=post, feedback=response.feedback)
        updated_rejected_posts = list(state.rejected_posts) + [rejected_entry]
        
        return {
            "qualified": False,
            "feedback": response.feedback,
            "rejected_posts": updated_rejected_posts,
            "retry_count": state.retry_count + 1
        }


## 5. Human-in-the-Loop (HITL) Review & Publishing


In [ ]:
def evaluation_router(state: State) -> Literal["human_review", "regenerate"]:
    """
    Routes after automated evaluation:
    - If post is qualified -> routes to human review.
    - If failed and under retry limit (3) -> routes back to regenerate.
    - If retry limit reached -> routes to human review for final guidance.
    """
    if state.qualified:
        return "human_review"
    
    if state.retry_count >= 5:
        print("[bold yellow]Maximum automated retries reached (5). Passing to human review for guidance.[/bold yellow]")
        return "human_review"

    print(f"[bold magenta]Routing back to post_generation_node for regeneration (Attempt {state.retry_count + 1})...[/bold magenta]")
    return "regenerate"


def human_review_node(state: State):
    """
    Step 4: Human-in-the-Loop Review Node.
    - Displays the complete post draft to the reviewer.
    - Uses LangGraph interrupt() to pause execution and await user approval or feedback.
    - On resumption, updates state with the decision.
    """
    print("\n" + "=" * 80)
    print(f"[bold cyan]       HUMAN-IN-THE-LOOP (HITL) REVIEW — ATTEMPT {state.retry_count + 1}[/bold cyan]")
    print("=" * 80)
    print("[bold green]✓ Automated evaluation passed. Here is the COMPLETE generated post for your review:[/bold green]\n")
    print("-" * 80)
    sys.stdout.write((state.post or "") + "\n")
    sys.stdout.flush()
    print("-" * 80 + "\n")
    
    # Pause graph execution and wait for human response
    review_data = interrupt({
        "action": "human_review_post",
        "post": state.post,
        "message": "The post has passed evaluation. Do you approve it for publishing?"
    })
    
    # Process resume payload from Command(resume=...)
    is_approved = review_data.get("approved", False) if isinstance(review_data, dict) else bool(review_data)
    
    if is_approved:
        print("[bold green]✓ Human approved the post![/bold green]\n")
        return {
            "human_approved": True,
            "feedback": None
        }
    else:
        human_feedback = review_data.get("feedback") if isinstance(review_data, dict) else "Post was rejected by human reviewer."
        if not human_feedback:
            human_feedback = "Post was rejected by human reviewer without specific feedback."
            
        print("[bold red]✗ Human rejected the post.[/bold red]")
        print(f"[bold yellow]Feedback noted for regeneration:[/bold yellow] {human_feedback}\n")
        
        rejected_entry = PostReview(post_generated=state.post, feedback=human_feedback)
        return {
            "human_approved": False,
            "feedback": human_feedback,
            "rejected_posts": list(state.rejected_posts) + [rejected_entry],
            "retry_count": state.retry_count + 1
        }


def human_review_router(state: State) -> Literal["upload_node", "post_generation_node"]:
    """
    Routes after human review:
    - If approved -> upload_node (prints and uploads).
    - If rejected -> post_generation_node (regenerates with human feedback).
    """
    if state.human_approved:
        return "upload_node"
    return "post_generation_node"


def upload_node(state: State):
    """
    Step 5: Final Upload and Publishing Node.
    Prints the approved post clearly and marks completion.
    """
    print("\n[bold green]================================================================================[/bold green]")
    print("[bold green]                     FINAL APPROVED POST PUBLISHED                              [/bold green]")
    print("[bold green]================================================================================[/bold green]\n")
    sys.stdout.write((state.post or "") + "\n\n")
    sys.stdout.flush()
    print("[bold green]✓ Post successfully approved by human and uploaded/printed![/bold green]")
    
    return {
        "messages": [AIMessage(content=f"Post uploaded successfully:\n\n{state.post}")]
    }


## 6. Graph Compilation & Visualization


In [ ]:
# Checkpointer for saving execution state across interrupts
checkpointer = InMemorySaver()

# Assemble the StateGraph
graph = StateGraph(State)

# 1. Register Nodes
graph.add_node("post_generation_node", post_generation_node)
graph.add_node("post_evaluator_node", post_evaluator_node)
graph.add_node("human_review_node", human_review_node)
graph.add_node("upload_node", upload_node)

# 2. Add Fixed Edges
graph.add_edge(START, "post_generation_node")
graph.add_edge("post_generation_node", "post_evaluator_node")

# 3. Add Conditional Routing
# From evaluator: pass -> human review; fail -> regenerate
graph.add_conditional_edges(
    "post_evaluator_node",
    evaluation_router,
    {
        "human_review": "human_review_node",
        "regenerate": "post_generation_node"
    }
)

# From human review: approved -> upload; rejected -> regenerate
graph.add_conditional_edges(
    "human_review_node",
    human_review_router,
    {
        "upload_node": "upload_node",
        "post_generation_node": "post_generation_node"
    }
)

# From upload to end
graph.add_edge("upload_node", END)

# Compile workflow with checkpointer
workflow = graph.compile(checkpointer=checkpointer)
print("[bold green]HITL Workflow graph compiled successfully with InMemorySaver checkpointer![/bold green]")


In [ ]:
from IPython.display import Image, display

# Display graph architecture visually
try:
    display(Image(workflow.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph visual rendering note: {e}")


## 7. Interactive Execution & Human-in-the-Loop Demo


In [ ]:
def respond_to_review(approved: bool, feedback: str | None = None, thread_id: str = "social-post-1"):
    """
    Resume an interrupted workflow execution from a separate notebook cell.
    
    Args:
        approved: True to approve and publish, False to reject.
        feedback: Constructive critique if rejected (optional if approved).
        thread_id: Session identifier to match the active conversation thread.
    """
    config = {"configurable": {"thread_id": thread_id}}
    return workflow.invoke(
        Command(resume={"approved": approved, "feedback": feedback}),
        config=config
    )


def run_workflow_with_human_loop(user_query: str, thread_id: str = "social-post-1"):
    """
    Runs the complete post generation and review workflow interactively.
    
    When the post qualifies, it displays the complete draft on screen
    and prompts the user for approval or feedback in the notebook.
    
    Args:
        user_query: The topic or query for the post.
        thread_id: Unique identifier for persisting session checkpoints.
    """
    config = {"configurable": {"thread_id": thread_id}}
    initial_state = State(user_query=user_query)
    
    result = workflow.invoke(initial_state, config=config)
    
    # Loop as long as the workflow encounters a human review interrupt
    while "__interrupt__" in result:
        interrupt_info = result["__interrupt__"][0].value
        post_draft = interrupt_info.get("post") or ""
        
        # Display the full post draft clearly before asking for approval
        print("\n" + "=" * 80)
        print("[bold cyan]               HUMAN ACTION REQUIRED: APPROVE OR REVISE POST[/bold cyan]")
        print("=" * 80)
        print("[bold yellow]Please review the complete draft below before deciding:[/bold yellow]\n")
        print("-" * 80)
        sys.stdout.write(post_draft + "\n")
        sys.stdout.flush()
        print("-" * 80 + "\n")
        
        choice = input("Do you approve this post? (yes/no): ").strip().lower()
        if choice in ["yes", "y", "true", "approve", "approved"]:
            print("\n[bold green]Approval recorded! Proceeding to upload/publish...[/bold green]\n")
            result = workflow.invoke(Command(resume={"approved": True}), config=config)
        else:
            feedback = input("Post rejected. Please enter your feedback for revision: ").strip()
            print(f"\n[bold magenta]Feedback submitted: '{feedback}'. Regenerating post with your feedback...[/bold magenta]\n")
            result = workflow.invoke(Command(resume={"approved": False, "feedback": feedback}), config=config)
            
    return result


In [ ]:
# Run the interactive HITL workflow
# You will be prompted in the notebook to approve or provide feedback when the post is ready
final_result = run_workflow_with_human_loop(
    user_query="Write a LinkedIn post about the impact of generative AI on software engineering productivity in 2026.",
    thread_id="post-session-demo"
)
